[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/at1r-nanobody-7T83/blob/main/notebooks/AT118i4h32_analysis.ipynb)

# Nanobody AT118i4h32 (PDB 7T83) — Structure & Dynamics
**AT1R antagonist nanobody · X-ray 2.10 Å · 8 × 128-residue chains · GROMACS MD**

Live companion to the project website: https://Harshini-Suresha.github.io/at1r-nanobody-7T83/

> Run top-to-bottom in Colab (Runtime → Run all). No API key needed. Sections 1–4 recompute the numbers shown on the website; Section 5 plots MD observables (with synthetic examples + hooks for your own `.xvg` files).


## 0. Setup
Installs Biopython (PDB parsing) if missing. numpy / matplotlib / requests come with Colab.

In [ ]:
import sys
try:
    import Bio
    print("Biopython", Bio.__version__)
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "biopython"])
    import Bio
    print("installed Biopython", Bio.__version__)

import math, requests
import numpy as np
import matplotlib.pyplot as plt
print("numpy", np.__version__)


## 1. Fetch PDB 7T83 metadata live from RCSB
Same source the website labels "From RCSB".

In [ ]:
PDB_ID = "7T83"
url = f"https://data.rcsb.org/rest/v1/core/entry/{PDB_ID}"
r = requests.get(url, timeout=30)
r.raise_for_status()
entry = r.json()

print("Title:", entry.get("struct", {}).get("title"))
print("Method:", entry.get("exptl", [{}])[0].get("method"))
print("Resolution:", entry.get("rcsb_entry_info", {}).get("resolution_combined"))
print("Deposited:", entry.get("rcsb_accession_info", {}).get("deposit_date"))
print("Polymer chains:", entry.get("rcsb_entry_info", {}).get("polymer_entity_count"))
print("Non-polymer (ligands):", [n.get("comp_id") for n in entry.get("nonpolymer_entities", [])] if "nonpolymer_entities" in entry else "see polymer section")

# cell constants
cell = entry.get("cell", {})
print(cell)
a, b, c = cell.get("length_a"), cell.get("length_b"), cell.get("length_c")
al, be, ga = cell.get("angle_alpha"), cell.get("angle_beta"), cell.get("angle_gamma")
print(f"a={a} b={b} c={c} alpha={al} beta={be} gamma={ga}")
print("space group:", entry.get("symmetry", {}).get("space_group_name_h_m"))
print("Z (asu per cell):", entry.get("symmetry", {}).get("nsymop", "(use 2 for P21)"))

# R factors
for s in entry.get("refine", []):
    print("R-work:", s.get("ls_R_factor_R_work"), "R-free:", s.get("ls_R_factor_R_free"))


## 2. Crystallographic checks — cell volume & mass
Standard monoclinic cell: $V = a b c \sin\beta$. Compares sequence mass vs deposited weight.

In [ ]:
# Deposited constants for 7T83 (also returned by the API above)
a, b, c = 80.11, 85.40, 84.44
beta_deg = 108.64
V = a * b * c * math.sin(math.radians(beta_deg))
print(f"Cell volume = {V:,.1f} Å³")

SEQ = "MAEVQLVESGGGLVQPGGSLRLSCAASGYIYRRYRMGWYRQAPGKGREFVAAISGGSSTNYADSVKGRFTISRDNSKNTVYLQMNSLRAEDTAVYYCAAYRIVSDPRVYWGQGTQVTVSSLEHHHHHH"
print("Length:", len(SEQ))

res_mass = {'G':57.0519,'A':71.0788,'S':87.0782,'P':97.1167,'V':99.1326,'T':101.1051,
 'C':103.1388,'L':113.1594,'I':113.1594,'N':114.1038,'D':115.0886,'Q':128.1307,
 'K':128.1741,'E':129.1155,'M':131.1926,'H':137.1411,'F':147.1766,'R':156.1875,
 'Y':163.176,'W':186.2132}
chain_mass = sum(res_mass[aa] for aa in SEQ) + 18.015  # + water
print(f"Chain mass (sequence) = {chain_mass:,.1f} Da = {chain_mass/1000:.2f} kDa")

asu_mass = 8*chain_mass + 4*189.11  # 8 chains + 4 citrate (FLC C6H8O7 ~189 Da)
print(f"ASU (8 chains + 4 citrate) = {asu_mass:,.1f} Da = {asu_mass/1000:.2f} kDa")
print("RCSB total weight ~114.56 kDa → 2 ASU per cell ≈", f"{2*asu_mass/1000:.2f} kDa (close; waters/ions omitted)")

r_work, r_free = 0.208, 0.257
print(f"R-free − R-work = {r_free-r_work:.3f} (unremarkable at 2.10 Å)")


## 3. Matthews coefficient — why 8 chains?
$V_M = V_{cell} / (M_{ASU} \times Z)$, solvent $= 1 - 1.23/V_M$. Usual protein range 1.7–3.5 Å³/Da.

In [ ]:
Z = 2  # P 1 21 1 has 2 asymmetric units per cell
print(f"{'n chains/ASU':>14} | {'Vm':>6} | {'solvent':>7} | verdict")
for nch in [1,2,4,6,8,12]:
    m_asu = nch*chain_mass  # protein only (citrate negligible for verdict)
    vm = V/(m_asu*Z)
    solv = 1 - 1.23/vm
    verdict = "OK ✓" if 1.7 <= vm <= 3.5 else "outside"
    print(f"{nch:>14} | {vm:>6.2f} | {solv:>6.1%} | {verdict}")

# plot
ns = np.array([1,2,4,6,8,12])
vms = np.array([V/(n*chain_mass*Z) for n in ns])
plt.figure()
plt.axhspan(1.7, 3.5, alpha=0.2)
plt.plot(ns, vms, marker="o")
for x,y in zip(ns, vms):
    plt.text(x, y+0.3, f"{y:.2f}", ha="center", fontsize=9)
plt.xlabel("chains per asymmetric unit")
plt.ylabel("Matthews Vm (Å³/Da)")
plt.title("Matthews check: only ~8 chains/ASU falls in the usual range")
plt.grid(alpha=0.3)
plt.show()
print("→ 8 chains/ASU gives Vm ≈ 2.41, solvent ≈ 49% — the deposited packing.")


## 4. Sequence & anatomy — 128 residues, CDRs, charge, hydropathy
Region boundaries are IMGT-style approximations (confirm with a numbering tool before quoting ranges).

In [ ]:
kd = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
      'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
      'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}

# Approximate regions for this construct (0-based, end-exclusive) — same split as website
regions = [
    ("FR1", 0, 25), ("CDR1", 25, 33), ("FR2", 33, 52),
    ("CDR2", 52, 60), ("FR3", 60, 98), ("CDR3", 98, 110),
    ("FR4", 110, 121), ("His-tag", 121, 128),
]
print(f"{'region':<8}{'range':<12}{'len':<5}{'seq':<16}{'charge':<8}hydropathy")
for name, s, e in regions:
    frag = SEQ[s:e]
    q = sum(1 for x in frag if x in "KR") - sum(1 for x in frag if x in "DE")
    h = sum(kd[x] for x in frag)/len(frag)
    print(f"{name:<8}{s+1}-{e:<10}{len(frag):<5}{frag:<16}{q:<8}{h:+.2f}")

print()
print("Full charge (K+R − D−E, His neutral):", sum(1 for x in SEQ if x in "KR") - sum(1 for x in SEQ if x in "DE"))
print("Cysteines at (1-based):", [i+1 for i,x in enumerate(SEQ) if x=="C"], "→ conserved disulfide C24–C97")
print("Histidines at (1-based):", [i+1 for i,x in enumerate(SEQ) if x=="H"], "→ all in C-terminal tag, no nanobody His to protonate")
print("CDR1:", SEQ[25:33], "| CDR2:", SEQ[52:60], "| CDR3:", SEQ[98:110])

# hydropathy profile
win = 5
prof = np.array([np.mean([kd[SEQ[j]] for j in range(max(0,i-win), min(len(SEQ), i+win+1))]) for i in range(len(SEQ))])
plt.figure(figsize=(10,3.5))
plt.plot(np.arange(1,129), prof)
for name,s,e in regions:
    if "CDR" in name:
        plt.axvspan(s+1, e, alpha=0.25)
plt.xlabel("residue number")
plt.ylabel(f"mean KD (window ±{win})")
plt.title("Hydropathy profile — shaded = CDR loops")
plt.grid(alpha=0.3)
plt.show()

# 16-per-row sequence map
print("128 residues, 16 per row (amber=CDR, rose=tag in website):")
for i in range(0,128,16):
    print(f"{i+1:>3}-{i+16:<3} {SEQ[i:i+16]}")


## 5. Download 7T83 & inspect in the notebook
Parses the PDB, counts chains/waters/hetero atoms — the "preparation" starting point.

In [ ]:
from Bio.PDB import PDBList, PDBParser
import os
pdbl = PDBList()
pdb_path = pdbl.retrieve_pdb_file("7T83", pdir=".", file_format="pdb")
print("downloaded:", pdb_path)
parser = PDBParser(QUIET=True)
st = parser.get_structure("7T83", pdb_path)
chains = list(st[0].get_chains())
print("chains:", [c.id for c in chains], "| residues in chain A:", len(list(chains[0].get_residues())))
# hetero / waters
hets, waters = set(), 0
for res in st[0].get_residues():
    het = res.get_id()[0]
    if het == "W_":
        waters += 1
    elif het != " ":
        hets.add(res.get_resname())
print("hetero resnames:", hets, "| waters:", waters)
print("Download PDB directly: https://files.rcsb.org/download/7T83.pdb")


## 6. MD observables — RMSD / RMSF / Rg / H-bonds
Replace the synthetic series below with your GROMACS output (`rmsd.xvg`, `rmsf.xvg`, `gyrate.xvg`, `hbond.xvg`).
In Colab: upload files via the Files pane, then set `USE_UPLOAD = True`.

In [ ]:
USE_UPLOAD = False  # ← set True after uploading your .xvg files
rng = np.random.default_rng(7)
t = np.arange(0, 100, 0.1)  # ns

def synth_rmsd(t):
    return 0.18 + 0.06*(1-np.exp(-t/12)) + 0.015*np.sin(t*0.7) + rng.normal(0, 0.008, len(t))

def synth_rg(t):
    return 1.52 + 0.015*(1-np.exp(-t/20)) + rng.normal(0, 0.004, len(t))

def synth_hb(t):
    return 95 + 6*np.sin(t*0.15) + rng.normal(0, 2.5, len(t))

def load_xvg(path, skip=24):
    d = []
    with open(path) as f:
        for line in f:
            if line.startswith(("#", "@")): continue
            d.append([float(x) for x in line.split()[:2]])
    return np.array(d)

if USE_UPLOAD:
    rmsd = load_xvg("rmsd.xvg"); rg = load_xvg("gyrate.xvg"); hb = load_xvg("hbond.xvg")
    fig, ax = plt.subplots(3,1, figsize=(9,7), sharex=True)
    ax[0].plot(rmsd[:,0], rmsd[:,1]); ax[0].set_ylabel("RMSD (nm)")
    ax[1].plot(rg[:,0], rg[:,1]);   ax[1].set_ylabel("Rg (nm)")
    ax[2].plot(hb[:,0], hb[:,1]);   ax[2].set_ylabel("# H-bonds"); ax[2].set_xlabel("time (ps)")
    plt.suptitle("Your GROMACS observables (uploaded .xvg)")
else:
    rmsd_y, rg_y, hb_y = synth_rmsd(t), synth_rg(t), synth_hb(t)
    prod = t >= 20  # example: discard first 20 ns equilibration
    print(f"Example convention: discard t<20 ns; production mean RMSD={rmsd_y[prod].mean():.3f}±{rmsd_y[prod].std():.3f} nm, "
          f"Rg={rg_y[prod].mean():.3f}±{rg_y[prod].std():.3f} nm, H-bonds={hb_y[prod].mean():.0f}±{hb_y[prod].std():.0f}")
    fig, ax = plt.subplots(3,1, figsize=(9,7), sharex=True)
    ax[0].plot(t, rmsd_y); ax[0].axvline(20, ls="--"); ax[0].set_ylabel("RMSD (nm)")
    ax[1].plot(t, rg_y);   ax[1].axvline(20, ls="--"); ax[1].set_ylabel("Rg (nm)")
    ax[2].plot(t, hb_y);   ax[2].axvline(20, ls="--"); ax[2].set_ylabel("# H-bonds"); ax[2].set_xlabel("time (ns)")
    plt.suptitle("ILLUSTRATIVE synthetic observables — replace with your trajectory")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# RMSF example: flexible CDR3 / termini pattern
res = np.arange(1,129)
rmsf = 0.08 + 0.05*np.exp(-((res-104)/6)**2) + 0.03*np.exp(-((res-29)/4)**2) + 0.04*(res>121) + rng.normal(0,0.008,len(res))
rmsf = np.clip(rmsf, 0.03, None)
plt.figure(figsize=(10,3.2))
plt.plot(res, rmsf)
plt.xlabel("residue"); plt.ylabel("RMSF (nm)")
plt.title("ILLUSTRATIVE RMSF — expect peaks in CDR loops / termini; use Cα, first production frame as reference")
plt.grid(alpha=0.3); plt.show()
print("Conventions to state in your report: fitted/calculated atom group, reference frame, PBC correction, production start, units + sampling interval, mean±spread.")


## 7. Box-size estimator (planning aid, mirrors website slider)
Assumes ~4 nm nanobody, ~17 nm³ volume, ~1,900 atoms; water 33.4 molecules/nm³; 0.15 M NaCl.

In [ ]:
def estimate(pad_nm=1.0, prot_d=4.0, prot_atoms=1900):
    edge = prot_d + 2*pad_nm
    v = edge**3
    waters = (v - 17) * 33.4
    atoms = prot_atoms + waters*3
    ion_pairs = waters * 0.15/55.5
    return edge, int(waters), int(atoms), int(ion_pairs)

for pad in [0.8, 1.0, 1.2, 1.5]:
    e, w, at, ip = estimate(pad)
    print(f"pad {pad:.1f} nm → edge {e:.1f} nm, waters ~{w:,}, atoms ~{at:,}, 0.15M ion pairs ~{ip}")
print("\nCost scales ~ volume: 1.5 nm pad ≈ %.1fx atoms of 1.0 nm pad" % (estimate(1.5)[2]/estimate(1.0)[2]))


## Links
- 🌐 Website: https://Harshini-Suresha.github.io/at1r-nanobody-7T83/
- 💻 Repo: https://github.com/Harshini-Suresha/at1r-nanobody-7T83
- 📓 This notebook (raw): https://raw.githubusercontent.com/Harshini-Suresha/at1r-nanobody-7T83/main/notebooks/AT118i4h32_analysis.ipynb
- 🧬 RCSB 7T83: https://www.rcsb.org/structure/7T83 · PDB: https://files.rcsb.org/download/7T83.pdb
